<a href="https://colab.research.google.com/github/amarjithanand/email-ai-classifier/blob/main/colab/email_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import Section**





In [1]:
from sentence_transformers import SentenceTransformer

In [2]:
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
import numpy as np

In [4]:
import pickle

In [5]:
import pandas as pd

In [6]:
from google.colab import auth

In [7]:
from googleapiclient.discovery import build

In [8]:
from google.auth import default

In [9]:
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google.colab import files
import os
import json


# **Load BERT Model**





In [10]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# **Data Storage**

In [11]:
texts = []
labels = []
embeddings = []

# **Add Training Function**

In [12]:
def add_example(text, label):
    emb = model.encode(text)

    texts.append(text)
    labels.append(label)
    embeddings.append(emb)

# **Training Model**

In [13]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/email_dataset_2200.csv")
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,subject,body,label
0,Let's catch up this weekend,Let's catch up this weekend. Kindly take neces...,Personal
1,HR follow up for application,HR follow up for application. Immediate attent...,Internship
2,Join our internship training program,Join our internship training program. Please r...,Promotion
3,Business proposal discussion,Business proposal discussion. Let me know your...,Opportunities
4,Startup collaboration request,Startup collaboration request. Immediate atten...,Opportunities


In [14]:
df["text"] = df["subject"] + " " + df["body"]

texts = df["text"].tolist()
labels = df["label"].tolist()

In [15]:
embeddings = model.encode(texts, batch_size=32, show_progress_bar=True)

Batches:   0%|          | 0/69 [00:00<?, ?it/s]

In [16]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def predict(text):
    new_emb = model.encode([text])

    sims = cosine_similarity(new_emb, embeddings)[0]

    idx = np.argmax(sims)

    return labels[idx], sims[idx]

In [17]:
def classify(text):
    label, score = predict(text)

    if score > 0.8:
        action = "auto"
    elif score > 0.5:
        action = "suggest"
    else:
        action = "llm"

    return label, score, action

In [18]:
print(classify("Client meeting scheduled tomorrow"))
print(classify("Huge discount on internship training course"))
print(classify("Invoice for your recent payment"))
print(classify("Happy birthday! Have a great day"))

('Work', np.float32(0.8281528), 'auto')
('Promotion', np.float32(0.58556587), 'suggest')
('Finance', np.float32(0.6978997), 'suggest')
('Personal', np.float32(0.7618115), 'suggest')


# **E-Mail Integration**

In [34]:
import os
import json
!pip install --quiet google-api-python-client google-auth-httplib2 google-auth-oauthlib

from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google.colab import files
from google.oauth2.credentials import Credentials # Import Credentials

# Upload credentials.json
print("Please upload your client_secrets.json file:")
uploaded = files.upload()

SCOPES = [
    "https://www.googleapis.com/auth/gmail.readonly",
    "https://www.googleapis.com/auth/gmail.modify"
]

client_secrets_file = list(uploaded.keys())[0]

# Create a flow object, specifying 'urn:ietf:wg:oauth:2.0:oob' as redirect_uri for console apps
flow = InstalledAppFlow.from_client_secrets_file(
    client_secrets_file, SCOPES, redirect_uri='urn:ietf:wg:oauth:2.0:oob')

# Generate the authorization URL
auth_url, _ = flow.authorization_url(prompt='consent')

print(f"Please go to this URL to authorize access: {auth_url}")

# Prompt the user to paste the authorization code
auth_code = input('Enter the authorization code from the browser here: ')

# Exchange the authorization code for an OAuth2Token object
oauth2_token = flow.fetch_token(code=auth_code)

# Convert OAuth2Token to google.oauth2.credentials.Credentials
creds = Credentials(
    token=oauth2_token['access_token'],
    refresh_token=oauth2_token.get('refresh_token'),
    token_uri=flow.client_config['token_uri'],
    client_id=flow.client_config['client_id'],
    client_secret=flow.client_config['client_secret'],
    scopes=SCOPES
)

service = build('gmail', 'v1', credentials=creds)

Please upload your client_secrets.json file:


Saving credentials.json to credentials (8).json
Please go to this URL to authorize access: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=164668447129-jrvnh3q5gim4hl3iqjjda6o9cash6fbs.apps.googleusercontent.com&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.modify&state=ZOKhy3rYxCxOLOEB3RprGvdbVGiGOm&code_challenge=FQkAwr2IhhAcEIBTPpKriLfR23P-HOGrFQOaU08Fhes&code_challenge_method=S256&prompt=consent&access_type=offline
Enter the authorization code from the browser here: 4/1AdkVLPzuKaoepmBmgWd-4x9e3uVDTcvLj46exEu9F_IzpeWKWNyTWCABirw


In [45]:

# Fetch labels
gmail_labels_response = service.users().labels().list(userId='me').execute()
user_labels = []

print("LABELS:")
for l in gmail_labels_response['labels']:
  if l['type'] == 'user':
        user_labels.append(l)
        print(l['name'])

LABELS:
Internship


In [48]:
messages = service.users().messages().list(userId='me', maxResults=10).execute()

print("\nEMAIL CLASSIFICATION:")

for i, msg in enumerate(messages.get('messages', []), start=1):
    full = service.users().messages().get(userId='me', id=msg['id']).execute()

    text = full.get('snippet', "").strip()

    if not text:
        continue

    label, score, action = classify(text)

    print(f"\n--- EMAIL {i} --------------------")
    print("EMAIL:", text[:150])
    print("PREDICTED:", label)
    print("CONFIDENCE:", round(score, 3))
    print("ACTION:", action)


EMAIL CLASSIFICATION:


KeyError: np.int64(1300)